In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/home/yuanshanwu/Documents/GitHub/QuantUS/engines/ceus/CLI-Demos
/home/yuanshanwu/Documents/GitHub/QuantUS/engines/ceus


## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['mp4', 'nifti', 'avi']


In [3]:
scan_type = 'nifti'

# Takes the DICOM file as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
}

In [4]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [5]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['load_bolus_mask', 'nifti']


In [6]:
seg_type = 'nifti'

seg_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/MotionCompensated_QUANTUSGUI.nii.gz'
seg_loader_kwargs = {}

In [7]:
from src.entrypoints import seg_loading_step

# Testing the motion compensation, right now is hard coded
seg_data = seg_loading_step(seg_type, image_data, seg_path, CEUS_scan_path, **seg_loader_kwargs)

# Visualize the Motion compensation outcome

# Implement MEDSAM 2 model

In [9]:
# Use inline backend - MORE RELIABLE for Jupyter
%matplotlib inline

import cv2
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import binary_erosion
import imageio
from PIL import Image
import matplotlib.patches as patches
from tqdm import tqdm

def enhance_bmode_noise(image_slice, p_low_percentile=2.0, p_high_percentile=98.0):
    non_zero = image_slice[image_slice != 0]
    # Compute 10th and 90th percentile on non-zero values
    p_low = np.percentile(non_zero, 2)
    p_high = np.percentile(non_zero, 98)

    # Clip and normalize to 0-255
    clipped_slice = np.clip(image_slice, p_low, p_high)
    normalized_slice = ((clipped_slice - p_low) / (p_high - p_low) * 255).astype(np.uint8)
    
    return normalized_slice


def get_mask_boundary(mask_slice):
    """Extract boundary of a binary mask using erosion."""
    if mask_slice.max() == 0:
        return np.zeros_like(mask_slice, dtype=bool)
    eroded = binary_erosion(mask_slice)
    boundary = mask_slice.astype(bool) & ~eroded
    return boundary

def get_voi_center(mask_3d):
    """Calculate the center of a 3D VOI mask."""
    coords = np.where(mask_3d > 0)
    if len(coords[0]) == 0:
        return None, None, None
    lateral_center = int(np.mean(coords[0]))
    depth_center = int(np.mean(coords[1]))
    elevation_center = int(np.mean(coords[2]))
    return lateral_center, depth_center, elevation_center

In [10]:
import sys
# Add MedSAM2 root to Python path
# From CLI-Demos/, go up 4 levels to QUANTUS/, then into MedSAM2/
medsam2_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'MedSAM2'))
print(f"MedSAM2 path: {medsam2_path}")  # verify it looks right
sys.path.insert(0, medsam2_path)

# load libraries and define necessary functions
from glob import glob
from tqdm import tqdm
import os
from os.path import join, basename
import re
import matplotlib.pyplot as plt
from collections import OrderedDict
import pandas as pd
import numpy as np
import argparse

from PIL import Image
import SimpleITK as sitk
import torch
import torch.multiprocessing as mp
from sam2.build_sam import build_sam2_video_predictor_npz
import SimpleITK as sitk
from skimage import measure, morphology

MedSAM2 path: /home/yuanshanwu/Documents/GitHub/QuantUS/MedSAM2


In [12]:
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Load model
checkpoint = os.path.join(medsam2_path, 'checkpoints', 'MedSAM2_2411.pt')
model_cfg = "configs/sam2.1_hiera_t512.yaml"

sam2_model = build_sam2(model_cfg, checkpoint)
image_predictor = SAM2ImagePredictor(sam2_model)

start_frame = 0
end_frame = 300
# Pick a frame and get the bbox
for frame_idx in tqdm(range(start_frame, end_frame), desc="Comparing the Motion compensated VOI vs MedSAM2 segmentation"):
    # Clear previous output and display current frame
    clear_output(wait=True)
    
    bbox = seg_data.motion_compensation.tracked_bboxes[frame_idx]
    volume = bmode_image_data.pixel_data[:, :, :, frame_idx]

    # Central Z-slice of the bounding box
    z_mid = int(np.floor((bbox.z_min + bbox.z_max) // 2))

    # Extract 2D slice — your axial view is transposed in the viz script
    slice_2d = volume[:, :, z_mid]  # (X, Y)

    # Normalize to uint8 [0, 255]
    slice_2d = slice_2d.T # Transpose to match visualization orientation
    s_min, s_max = slice_2d.min(), slice_2d.max()
    slice_uint8 = ((slice_2d - s_min) / (s_max - s_min + 1e-8) * 255).astype(np.uint8)

    # Convert grayscale → RGB (MedSAM2 expects 3 channels)
    slice_rgb = np.stack([slice_uint8] * 3, axis=-1)  # (X, Y, 3)

    # 2D bounding box: [x_min, y_min, x_max, y_max]
    bbox_2d = np.array([bbox.x_min, bbox.y_min, bbox.x_max, bbox.y_max], dtype=np.float32)

    # Run inference
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        image_predictor.set_image(slice_rgb)
        masks, scores, logits = image_predictor.predict(
            point_coords=None,
            point_labels=None,
            box=bbox_2d[None, :],  # shape (1, 4)
            multimask_output=False,
        )

    pred_mask = masks[0]  # (H, W) binary mask

    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Original with bbox
    axes[0].imshow(enhance_bmode_noise(slice_2d), cmap='gray')
    rect = patches.Rectangle(
        (bbox.x_min, bbox.y_min), bbox.x_max - bbox.x_min, bbox.y_max - bbox.y_min,
        edgecolor='yellow', facecolor='none', linewidth=2
    )
    axes[0].add_patch(rect)
    axes[0].set_title('Input + Bounding Box')

    # MedSAM2 prediction
    axes[1].imshow(enhance_bmode_noise(slice_2d), cmap='gray')
    axes[1].contour(pred_mask, colors='lime', linewidths=2)
    axes[1].set_title('MedSAM2 Segmentation')

    # Your existing MC mask for comparison
    mc_mask = seg_data.motion_compensation.apply_to_mask(seg_data.seg_mask, frame_idx, 0)
    axes[2].imshow(enhance_bmode_noise(slice_2d), cmap='gray')
    axes[2].contour(get_mask_boundary(mc_mask[:, :, z_mid].T), colors='red', linewidths=2)
    axes[2].contour(pred_mask, colors='lime', linewidths=2, linestyles='dashed')
    axes[2].set_title('Comparison: MC mask (red) vs MedSAM2 (green)')

    plt.suptitle(f'Frame {frame_idx}, Z-slice {z_mid}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

Comparing the Motion compensated VOI vs MedSAM2 segmentation:   3%|▎         | 8/300 [00:03<02:09,  2.26it/s]


KeyboardInterrupt: 

In [18]:
import torch
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from tqdm import tqdm
from IPython.display import clear_output

# Load model
checkpoint = os.path.join(medsam2_path, 'checkpoints', 'MedSAM2_MRI_LiverLesion.pt')
model_cfg = "configs/sam2.1_hiera_t512.yaml"
sam2_model = build_sam2(model_cfg, checkpoint)
image_predictor = SAM2ImagePredictor(sam2_model)

def slice_to_rgb(slice_2d):
    """Normalize a 2D slice to uint8 RGB for MedSAM2."""
    s_min, s_max = slice_2d.min(), slice_2d.max()
    uint8 = ((slice_2d - s_min) / (s_max - s_min + 1e-8) * 255).astype(np.uint8)
    return np.stack([uint8] * 3, axis=-1)

def run_medsam2(image_predictor, slice_rgb, bbox_2d):
    """Run MedSAM2 on a single 2D slice with a bounding box prompt."""
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        image_predictor.set_image(slice_rgb)
        masks, scores, logits = image_predictor.predict(
            point_coords=None,
            point_labels=None,
            box=bbox_2d[None, :],  # (1, 4)
            multimask_output=False,
        )
    return masks[0]  # (H, W) binary mask

start_frame = 0
end_frame = 300

for frame_idx in tqdm(range(start_frame, end_frame), desc="MedSAM2 3-plane segmentation"):
    clear_output(wait=True)

    bbox  = seg_data.motion_compensation.tracked_bboxes[frame_idx]
    volume = bmode_image_data.pixel_data[:, :, :, frame_idx]  # (X, Y, Z)
    mc_mask = seg_data.motion_compensation.apply_to_mask(seg_data.seg_mask, frame_idx, 0)  # (X, Y, Z)

    # ------------------------------------------------------------------ #
    # Derive slice centres from MC mask (more reliable than bbox midpoint)#
    # ------------------------------------------------------------------ #
    z_mid = np.argmax(mc_mask.sum(axis=(0, 1)))   # elevation slice with most voxels
    y_mid = np.argmax(mc_mask.sum(axis=(0, 2)))   # depth slice
    x_mid = np.argmax(mc_mask.sum(axis=(1, 2)))   # lateral slice

    # ================================================================== #
    # AXIAL VIEW  –  volume[:, :, z_mid] → transpose to (Y, X)           #
    # bbox: x→col (lateral), y→row (depth)  ✓ already correct after .T   #
    # ================================================================== #
    axial_slice = volume[:, :, z_mid].T                          # (Y, X)
    axial_rgb   = slice_to_rgb(axial_slice)
    axial_bbox  = np.array([bbox.x_min, bbox.y_min,
                            bbox.x_max, bbox.y_max], dtype=np.float32)
    axial_pred  = run_medsam2(image_predictor, axial_rgb, axial_bbox)
    axial_mc    = mc_mask[:, :, z_mid].T                         # (Y, X)

    # ================================================================== #
    # CORONAL VIEW  –  volume[:, y_mid, :] → shape (X, Z), no transpose  #
    # image axes: col=Z (elevation), row=X (lateral)                      #
    # bbox prompt: x→col=Z, y→row=X                                       #
    # ================================================================== #
    coronal_slice = volume[:, y_mid, :]                          # (X, Z)
    coronal_rgb   = slice_to_rgb(coronal_slice)
    coronal_bbox  = np.array([bbox.z_min, bbox.x_min,
                              bbox.z_max, bbox.x_max], dtype=np.float32)
    coronal_pred  = run_medsam2(image_predictor, coronal_rgb, coronal_bbox)
    coronal_mc    = mc_mask[:, y_mid, :]                         # (X, Z)

    # ================================================================== #
    # SAGITTAL VIEW  –  volume[x_mid, :, :] → shape (Y, Z), no transpose #
    # image axes: col=Z (elevation), row=Y (depth)                        #
    # bbox prompt: x→col=Z, y→row=Y                                       #
    # ================================================================== #
    sagittal_slice = volume[x_mid, :, :]                         # (Y, Z)
    sagittal_rgb   = slice_to_rgb(sagittal_slice)
    sagittal_bbox  = np.array([bbox.z_min, bbox.y_min,
                               bbox.z_max, bbox.y_max], dtype=np.float32)
    sagittal_pred  = run_medsam2(image_predictor, sagittal_rgb, sagittal_bbox)
    sagittal_mc    = mc_mask[x_mid, :, :]                        # (Y, Z)

    # ================================================================== #
    # PLOT  –  3 rows × 3 cols                                            #
    #   col 0: input + bbox                                               #
    #   col 1: MedSAM2 overlay                                            #
    #   col 2: MC mask (red) vs MedSAM2 (green dashed)                   #
    # ================================================================== #
    fig, axes = plt.subplots(3, 3, figsize=(18, 18))

    views = [
        {
            "label":  f"Axial  (Z={z_mid})",
            "xlabel": "Lateral (X)",
            "ylabel": "Depth (Y)",
            "slice":  axial_slice,
            "pred":   axial_pred,
            "mc":     axial_mc,
            "rect":   patches.Rectangle(
                          (bbox.x_min, bbox.y_min),
                          bbox.x_max - bbox.x_min,
                          bbox.y_max - bbox.y_min,
                          edgecolor='yellow', facecolor='none', linewidth=2),
        },
        {
            "label":  f"Coronal (Y={y_mid})",
            "xlabel": "Elevation (Z)",
            "ylabel": "Lateral (X)",
            "slice":  coronal_slice,
            "pred":   coronal_pred,
            "mc":     coronal_mc,
            "rect":   patches.Rectangle(
                          (bbox.z_min, bbox.x_min),
                          bbox.z_max - bbox.z_min,
                          bbox.x_max - bbox.x_min,
                          edgecolor='yellow', facecolor='none', linewidth=2),
        },
        {
            "label":  f"Sagittal (X={x_mid})",
            "xlabel": "Elevation (Z)",
            "ylabel": "Depth (Y)",
            "slice":  sagittal_slice,
            "pred":   sagittal_pred,
            "mc":     sagittal_mc,
            "rect":   patches.Rectangle(
                          (bbox.z_min, bbox.y_min),
                          bbox.z_max - bbox.z_min,
                          bbox.y_max - bbox.y_min,
                          edgecolor='yellow', facecolor='none', linewidth=2),
        },
    ]

    for row, v in enumerate(views):
        img_disp = enhance_bmode_noise(v["slice"])

        # --- col 0: input + bbox ---
        axes[row, 0].imshow(img_disp, cmap='gray')
        axes[row, 0].add_patch(v["rect"])
        axes[row, 0].set_title(f'{v["label"]}  |  Input + BBox')
        axes[row, 0].set_xlabel(v["xlabel"])
        axes[row, 0].set_ylabel(v["ylabel"])

        # --- col 1: MedSAM2 overlay ---
        axes[row, 1].imshow(img_disp, cmap='gray')
        axes[row, 1].contour(v["pred"], colors='lime', linewidths=2)
        axes[row, 1].set_title(f'{v["label"]}  |  MedSAM2')
        axes[row, 1].set_xlabel(v["xlabel"])
        axes[row, 1].set_ylabel(v["ylabel"])

        # --- col 2: MC mask (red) vs MedSAM2 (green dashed) ---
        axes[row, 2].imshow(img_disp, cmap='gray')
        axes[row, 2].contour(get_mask_boundary(v["mc"].astype(bool)),
                             colors='red', linewidths=2)
        axes[row, 2].contour(v["pred"],
                             colors='lime', linewidths=2, linestyles='dashed')
        axes[row, 2].set_title(f'{v["label"]}  |  MC (red) vs SAM2 (green)')
        axes[row, 2].set_xlabel(v["xlabel"])
        axes[row, 2].set_ylabel(v["ylabel"])

    fig.suptitle(f'Frame {frame_idx}  |  Axial Z={z_mid}  Coronal Y={y_mid}  Sagittal X={x_mid}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

MedSAM2 3-plane segmentation:  65%|██████▌   | 195/300 [02:57<01:35,  1.10it/s]


KeyboardInterrupt: 